In [1]:
from math import pi
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram, plot_bloch_multivector

In [2]:
def qft_rotations(circuit, n):
    if n == 0:
        return circuit
    n -= 1
    circuit.h(n)                              
    for qubit in range(n):
        circuit.cp(pi / 2**(n - qubit), qubit, n)
    qft_rotations(circuit, n)
    return circuit

In [ ]:
def swap_registers(circuit, n):
    for qubit in range(n // 2):
        circuit.swap(qubit, n - qubit - 1)
    return circuit
def qft(circuit, n):
    qft_rotations(circuit, n)
    swap_registers(circuit, n)
    return circuit

# QFT|000> = |+++>
qc = QuantumCircuit(3); qft(qc, 3)
sv = Aer.get_backend('statevector_simulator')
state = sv.run(transpile(qc, sv)).result().get_statevector(qc, decimals=3)
print(state)

In [ ]:
# QFT|101>
qc = QuantumCircuit(3)
qc.x(0); qc.x(2)                              # |101> (5)
qft(qc, 3)
state5 = sv.run(transpile(qc, sv)).result().get_statevector(qc, decimals=3)
plot_bloch_multivector(state5)

In [ ]:
# Inverse QFT: build QFT|5> by hand, then undo it
qc = QuantumCircuit(3)
for q in range(3): qc.h(q)
qc.p(5 * pi / 4, 0); qc.p(5 * pi / 2, 1); qc.p(5 * pi, 2)   # phases 2π·5·2^j/8
inv = qft(QuantumCircuit(3), 3).inverse()
qc.compose(inv, inplace=True)
qc.measure_all()
qasm = Aer.get_backend('qasm_simulator')
counts = qasm.run(transpile(qc, qasm), shots=1000).result().get_counts()
print(counts)                                 # {'101': 1000}  (5 in binary)
plot_histogram(counts)